In [1]:
import os 

os.chdir('../..')
os.getcwd()

'/home/leostre/Рабочий стол/SoftStairs-QAT'

In [2]:
from ultralytics import YOLO 

MODEL = 'yolo26n.pt'
DATASET = "HomeObjects-3K.yaml"

N_EPOCHS = 40
IMGSIZE = 640

# Check firing

Okay if @@@ INIT for convolutions

In [ ]:
from softstairs_qat import SoftStairsQuantizer, QuantizationConfig


def get_qconfig(strategy, t_start, steps, n_bits=8):
    return QuantizationConfig(
        n_bits=n_bits, normalized=True, t_scheduler_strategy=strategy, t_start=t_start, t_end=1e-4, n_steps=steps
    )

model = YOLO(MODEL)
qconfig = get_qconfig('linear', .1, 2, 4)
quantizer = SoftStairsQuantizer(model, qconfig, excluded_modules={n for n, p in model.named_modules() if 'bn' in n}, verbose=True)

Okay if @@@ FIRED for convolutions

In [ ]:
import torch 

model(torch.randn(1, 3, 640, 640))

Okay of convolutions with `_orig`

In [ ]:
[n for n, p in quantizer.model.named_parameters()]

In [ ]:
assert len(quantizer._hook_handles), 'no handles found'

In [ ]:
print([p[0] for p in model.named_buffers()])

# baseline no-qat

In [ ]:
model = YOLO(MODEL)
results = model.train(
    data=DATASET,
    epochs=N_EPOCHS,
    imgsz=IMGSIZE,
    save_period=10,
    name='no-qat'
)

# Baseline QAT

In [ ]:
from ultralytics.models.yolo.detect import DetectionTrainer
import torch
from torchao.quantization import quantize_, Int8WeightOnlyConfig
from torchao.quantization.qat import QATConfig
from functools import partial

class TorchQATTrainer(DetectionTrainer):
        def __init__(self, *args, qconfig=None, **kwargs):
             super().__init__(*args, **kwargs)
             if qconfig is None:
                  raise ValueError('qconfig not specified')
             self.qconfig = qconfig 

        def get_model(self, cfg=None, weights=None, verbose=True):
            model = super().get_model(cfg, weights, verbose)

            # QAT initialization happens HERE,
            # after the final training model exists.
            quantize_(model, QATConfig(self.qconfig, step="prepare"))

            return model
    

## Int8

In [ ]:


model = YOLO(MODEL)
int8wo = Int8WeightOnlyConfig(group_size=32)


results = model.train(
    data=DATASET,
    epochs=N_EPOCHS,
    imgsz=IMGSIZE,
    name=f'qat-Int8-{N_EPOCHS}e',
    save_period=10, 
    trainer=partial(TorchQATTrainer, qconfig=int8wo)
)


## Int4

In [ ]:
from torchao.quantization import quantize_, Int4WeightOnlyConfig
import torch
from torchao.quantization import quantize_, Int8WeightOnlyConfig
from torchao.quantization.qat import QATConfig


model = YOLO(MODEL)

int4wo = Int4WeightOnlyConfig(group_size=32)



results = model.train(
    data=DATASET,
    epochs=N_EPOCHS,
    imgsz=IMGSIZE,
    name=f'qat-Int4-{N_EPOCHS}e',
    save_period=10,
    trainer=partial(TorchQATTrainer, qconfig=int4wo)
)

# Running

In [3]:
from itertools import product 

STRATEGIES = ['linear',
               'cos', 'exp', 
            #    'constant', 
               ]
T_START = [.9, 0.5, 0.1]

In [4]:
import matplotlib.pyplot as plt 
from functools import partial
from softstairs_qat import SoftStairsQuantizer, QuantizationConfig



def get_qconfig(strategy, t_start, steps, n_bits, naive):
    return QuantizationConfig(
        n_bits=n_bits,
        normalized=True,
        t_scheduler_strategy=strategy,
        t_start=t_start,
        t_end=1e-4,
        n_steps=steps * BATCH_SIZE,
        naive=naive
    )


def run_experiment_nb(model_name, n_epochs, strategy, t_start, n_bits, naive):

    qconfig = get_qconfig(
        strategy,
        t_start,
        n_epochs,
        n_bits,
        naive
    )

    from ultralytics.models.yolo.detect import DetectionTrainer

    class SSQATTrainer(DetectionTrainer):
        def _build_train_pipeline(self):
            import math 
            from ultralytics.utils import LOCAL_RANK
            """Build dataloaders, optimizer, and scheduler for current batch size."""
            batch_size = self.batch_size // max(self.world_size, 1)
            self.train_loader = self.get_dataloader(
                self.data["train"], batch_size=batch_size, rank=LOCAL_RANK, mode="train"
            )
            final_batch_size = len(self.train_loader.sampler) % self.train_loader.batch_size or self.train_loader.batch_size
            if self.args.imgsz < 2 * self.stride and not self.train_loader.drop_last and final_batch_size == 1:
                raise ValueError(
                    f"final batch=1 training at imgsz={self.args.imgsz} gives BatchNorm a single value per channel; "
                    f"change batch or use imgsz >= {2 * self.stride}"
                )
            # Note: When training DOTA dataset, double batch size could get OOM on images with >2000 objects.
            self.test_loader = self.get_dataloader(
                self.data.get("val") or self.data.get("test"),
                batch_size=batch_size if self.args.task in {"obb", "semantic", "depth"} else batch_size * 2,
                rank=LOCAL_RANK,
                mode="val",
            )
            self.accumulate = max(round(self.args.nbs / self.batch_size), 1)  # accumulate loss before optimizing
            weight_decay = 0.
            iterations = math.ceil(len(self.train_loader.dataset) / max(self.batch_size, self.args.nbs)) * self.epochs
            self.optimizer = self.build_optimizer(
                model=self.model,
                name=self.args.optimizer,
                lr=self.args.lr0,
                momentum=self.args.momentum,
                decay=weight_decay,
                iterations=iterations,
            )
            self._setup_scheduler()

        def get_model(self, cfg=None, weights=None, verbose=True):
            model = super().get_model(cfg, weights, verbose)

            excluded_modules = {
                n for n, p in model.named_modules()
                if "bn" in n
            }

            self.quantizer = SoftStairsQuantizer(
                model,
                qconfig,
                excluded_modules=excluded_modules,
            )

            return model
        
        def optimizer_step(self):
            """
            Custom optimizer step for SSQAT training.
            
            Args:
                epoch: Current epoch number
                batch: Current batch index
                optimizer: The optimizer being used
                loss: The loss value from the current batch
            """
            # Call the parent optimizer_step first
            super().optimizer_step()
            if self.quantizer is not None:
                self.quantizer.step()

def run_experiment_nb(model_name, n_epochs, strategy, t_start, n_bits=8):
    model = YOLO(model_name) 
    qconfig = get_qconfig(strategy, t_start, n_epochs, n_bits)
    from ultralytics.models.yolo.detect import DetectionTrainer

    class QATTrainer(DetectionTrainer):

        def get_model(self, cfg=None, weights=None, verbose=True):
            model = super().get_model(cfg, weights, verbose)

            # QAT initialization happens HERE,
            # after the final training model exists.
            self.quantizer = SoftStairsQuantizer(model, qconfig, excluded_modules={n for n, p in model.named_modules() if 'bn' in n})

            return model
    
    model.train(data=DATASET, epochs=n_epochs, name=f'{strategy}-{t_start}-{qconfig.n_bits}b-{n_epochs}e', imgsz=IMGSIZE, trainer=QATTrainer,
    save_period=10 )
    # assert quantizer_storage[0] is not None, 'No Quantizer were Initiated'


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


In [7]:
import subprocess
import sys

for i, (t, strat) in enumerate(product(T_START, 
                                       STRATEGIES, 
                                       )):
    try:
        print("Running:", strat, t)

        result = subprocess.run(
            [
                sys.executable,
                "run_training.py",
                "--model", MODEL,
                "--epochs", str(N_EPOCHS),
                "--strategy", strat,
                "--t-start", str(t),
                "--bits", "4",
                "--naive", "1"
            ],
            cwd='/home/leostre/Рабочий стол/SoftStairs-QAT',
            check=True,
        )
    except KeyboardInterrupt:
        raise
    except:
        with open('/home/leostre/Рабочий стол/SoftStairs-QAT/experiments/yolo/sdout.txt', 'a') as file:
            print('PASSED', strat, t, file=file)

Running: linear 0.9


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


Running: strategy=linear, t_start=0.9, bits=4, epochs=40naive: True
New https://pypi.org/project/ultralytics/8.4.127 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.107 🚀 Python-3.10.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5080, 15832MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=HomeObjects-3K.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=N

Traceback (most recent call last):
  File "/home/leostre/Рабочий стол/SoftStairs-QAT/run_training.py", line 263, in <module>
    run_experiment_nb(
  File "/home/leostre/Рабочий стол/SoftStairs-QAT/run_training.py", line 232, in run_experiment_nb
    model.train(
  File "/home/leostre/Рабочий стол/SoftStairs-QAT/.venv/lib/python3.10/site-packages/ultralytics/engine/model.py", line 865, in train
    self.trainer.train()
  File "/home/leostre/Рабочий стол/SoftStairs-QAT/.venv/lib/python3.10/site-packages/ultralytics/engine/trainer.py", line 240, in train
    self._do_train()
  File "/home/leostre/Рабочий стол/SoftStairs-QAT/.venv/lib/python3.10/site-packages/ultralytics/engine/trainer.py", line 577, in _do_train
    self.metrics, self.fitness = self.validate()
  File "/home/leostre/Рабочий стол/SoftStairs-QAT/.venv/lib/python3.10/site-packages/ultralytics/engine/trainer.py", line 846, in validate
    metrics = self.validator(self)
  File "/home/leostre/Рабочий стол/SoftStairs-QAT/.venv/l

Running: cos 0.9


...

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


Running: strategy=exp, t_start=0.9, bits=4, epochs=40naive: True
New https://pypi.org/project/ultralytics/8.4.127 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.107 🚀 Python-3.10.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5080, 15832MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=HomeObjects-3K.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None

Exception in thread Thread-3 (_pin_memory_loop):
Traceback (most recent call last):
  File "/usr/lib/python3.10/threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.10/threading.py", line 953, in run
    self._target(*self._args, **self._kwargs)
  File "/home/leostre/Рабочий стол/SoftStairs-QAT/.venv/lib/python3.10/site-packages/torch/utils/data/_utils/pin_memory.py", line 52, in _pin_memory_loop
    do_one_step()
  File "/home/leostre/Рабочий стол/SoftStairs-QAT/.venv/lib/python3.10/site-packages/torch/utils/data/_utils/pin_memory.py", line 28, in do_one_step
    r = in_queue.get(timeout=MP_STATUS_CHECK_INTERVAL)
  File "/usr/lib/python3.10/multiprocessing/queues.py", line 122, in get
    return _ForkingPickler.loads(res)
  File "/home/leostre/Рабочий стол/SoftStairs-QAT/.venv/lib/python3.10/site-packages/torch/multiprocessing/reductions.py", line 540, in rebuild_storage_fd
    fd = df.detach()
  File "/usr/lib/python3.10/multiprocessing/resource_share

WARNING ⚠️ 
WARNING ⚠️ MLflow: Failed to initialize: The filesystem tracking backend (e.g., './mlruns') is in maintenance mode and will not receive further updates. Please migrate to a database backend (e.g., 'sqlite:///mlflow.db') to access the latest MLflow features. The `mlflow migrate-filestore` tool migrates your existing data losslessly. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance. If the filesystem backend is required for your workflow, set `MLFLOW_ALLOW_FILE_STORE=true` to opt out of this exception.
WARNING ⚠️ MLflow: Not tracking this run
Image sizes 640 train, 640 val
Using 8 dataloader workers
Logging results to /home/leostre/Рабочий стол/SoftStairs-QAT/runs/detect/exp-0.9-4b-40e
Starting training for 40 epochs...
: 0% ──────────── 0/36  0.0s


KeyboardInterrupt: 

In [ ]:
from itertools import product 


for strat, t in product(
    # ['exp'], [5e-1]
    STRATEGIES, T_START
    ):
    if strat == 'linear' and t == 0.5:
        continue
    print('Running:', strat, t)
    run_experiment_nb(MODEL, n_epochs=2, strategy=strat, t_start=t, n_bits=4)
    